# Build a ClawBio Skill

Create a complete skill from scratch: SKILL.md contract, Python implementation, demo data, and validation.

| | |
|---|---|
| **Level** | Intermediate |
| **Time** | ~30 min |
| **Requirements** | Google account |
| **Full tutorial** | [docs.clawbio.ai/tutorials/build-a-skill](https://docs.clawbio.ai/tutorials/build-a-skill/) |

## Step 0: Setup

In [ ]:
!git clone --depth 1 https://github.com/ClawBio/ClawBio.git
%cd ClawBio
!pip install -q -r requirements.txt pyyaml
print("\n✓ ClawBio installed successfully")

## Step 1: Create the Skill Directory

In [ ]:
!mkdir -p skills/my-awesome-skill
print("✓ Directory created: skills/my-awesome-skill/")

## Step 2: Write the SKILL.md
The SKILL.md is the contract. It defines what the skill does, its inputs, outputs, domain decisions, and safety rules.

In [ ]:
%%writefile skills/my-awesome-skill/SKILL.md
---
name: my-awesome-skill
version: 0.1.0
author: Your Name <you@example.com>
domain: genomics
description: Analyse a set of variants and produce a summary report
license: MIT

inputs:
  - name: input_file
    type: file
    format: [vcf, csv, tsv, txt]
    description: Input data file
    required: true

outputs:
  - name: report
    type: file
    format: md
    description: Analysis report

dependencies:
  python: ">=3.11"
  packages:
    - pandas>=2.0

tags: [genomics, variants, tutorial]

demo_data:
  - path: demo_input.txt
    description: Synthetic test data with 3 variants

endpoints:
  cli: python skills/my-awesome-skill/my_skill.py --input {input_file} --output {output_dir}
---

## Domain Decisions

- **Threshold for significance**: p < 0.05 after Bonferroni correction
- **Reference database**: ClinVar (release 2025-12)
- **Population frequency filter**: exclude variants with gnomAD AF > 0.01

## Safety Rules

- Never report a clinical diagnosis; always include the research-use disclaimer
- Do not extrapolate results across ancestries unless validated

## Agent Boundary

The agent (LLM) dispatches and explains. The skill (Python) executes.
The agent must NOT override thresholds or invent associations.

## Step 3: Add Demo Data

In [ ]:
%%writefile skills/my-awesome-skill/demo_input.txt
# Synthetic demo data for my-awesome-skill
# This is NOT real patient data
rsid	chromosome	position	genotype
rs1234567	1	12345678	AG
rs2345678	2	23456789	CC
rs3456789	3	34567890	TT

## Step 4: Write the Python Implementation

In [ ]:
%%writefile skills/my-awesome-skill/my_skill.py
#!/usr/bin/env python3
"""My Awesome Skill — analyse variants and produce a summary report."""

import argparse
import json
from pathlib import Path

import pandas as pd


def run(input_path: Path, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(input_path, sep="\t", comment="#")
    print(f"✓ Loaded {len(df)} variants from {input_path.name}")

    # --- Your analysis logic here ---
    results = {"variants_loaded": len(df), "status": "complete"}

    report_path = output_dir / "report.md"
    report_path.write_text(
        f"# My Awesome Skill Report\n\n"
        f"Processed **{len(df)} variants**.\n\n"
        f"| rsID | Chromosome | Genotype |\n"
        f"|------|-----------|----------|\n"
        + "".join(f"| {r.rsid} | {r.chromosome} | {r.genotype} |\n" for r in df.itertuples())
        + f"\n*ClawBio is a research tool. Not a medical device.*\n"
    )
    print(f"✓ Report written to {report_path}")

    summary_path = output_dir / "summary.json"
    summary_path.write_text(json.dumps(results, indent=2))
    print(f"✓ Summary written to {summary_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="My Awesome Skill")
    parser.add_argument("--input", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    args = parser.parse_args()
    run(args.input, args.output)

## Step 5: Test It

In [ ]:
!python3 skills/my-awesome-skill/my_skill.py \
  --input skills/my-awesome-skill/demo_input.txt \
  --output /tmp/my-skill-demo

In [ ]:
from IPython.display import Markdown
Markdown(open('/tmp/my-skill-demo/report.md').read())

In [ ]:
import json
with open('/tmp/my-skill-demo/summary.json') as f:
    print(json.dumps(json.load(f), indent=2))

## Step 6: Validate Your SKILL.md

In [ ]:
import yaml
from pathlib import Path

skill_path = Path('skills/my-awesome-skill/SKILL.md')
text = skill_path.read_text()
front = text.split('---')[1]
meta = yaml.safe_load(front)

required = ['name', 'version', 'author', 'domain', 'description', 'inputs', 'outputs']
missing = [f for f in required if f not in meta]
if missing:
    print(f'❌ Missing fields: {missing}')
else:
    print(f'✓ SKILL.md valid: {meta["name"]} v{meta["version"]}')

## Done!

You have created a complete ClawBio skill with:
- A SKILL.md contract
- A Python implementation
- Synthetic demo data
- Validation

To submit your skill to ClawBio, fork the repository on GitHub, add your skill directory, and open a pull request.

---

**Next:** [Variant Interpretation Workshop](https://docs.clawbio.ai/tutorials/variant-interpretation-workshop/) | [GWAS Workshop](https://docs.clawbio.ai/tutorials/gwas-workshop/)

*ClawBio is a research and educational tool. It is not a medical device.*